# NLP Mid-Semester Exam — All Questions (AI357)

**SVNIT Surat · NLP · Monsoon 2026**

This notebook contains worked, executable solutions for all five questions in the uploaded question paper:
1. English noun morphology using a finite-state transducer (FST)
2. WordPiece tokenization with two merges
3. TF-IDF vectors and closest sentence pair
4. Naive Bayes spam classification for email E8 = (0, 0, 0)
5. Kneser–Ney probabilities with discount `d = 0.5`

Run the cells from top to bottom. Calculations and tables are generated with Python so you can inspect the intermediate values.

**Conventions where the question leaves details unspecified**
- Q2 uses the WordPiece-style normalized pair score `count(pair) / (count(left) * count(right))`; ties are broken deterministically.
- Q3 uses natural logarithm IDF: `idf(t) = ln(N / df(t))`, with raw term counts for TF and no smoothing.
- Q4 first shows unsmoothed Bernoulli Naive Bayes, then explains the zero-probability issue. No smoothing is specified in the question.
- Q5 uses standard interpolated Kneser–Ney bigram smoothing and no sentence-boundary markers, as instructed.

## Q1. Finite-State Transducer for English noun singular/plural

### Design

The transducer maps a noun stem to a lexical form with grammatical features. Singular nouns receive `+N+SG`. For plural:
- regular nouns append `s`
- nouns ending in `s`, `x`, `z`, `ch`, or `sh` append `es`

A practical FST can be represented with a stem-processing state and suffix-selection states. The table below describes the suffix behavior; the stem itself is copied unchanged by the identity transitions.

In [1]:
import pandas as pd

fst_transitions = pd.DataFrame([
    {"Current state": "q0", "Input / condition": "End of noun; suffix does NOT end in s, x, z, ch, sh", "Output": "copy stem + s +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "End of noun; suffix ends in s", "Output": "copy stem + es +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "End of noun; suffix ends in x", "Output": "copy stem + es +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "End of noun; suffix ends in z", "Output": "copy stem + es +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "End of noun; suffix ends in ch", "Output": "copy stem + es +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "End of noun; suffix ends in sh", "Output": "copy stem + es +N+PL", "Next state": "qf"},
    {"Current state": "q0", "Input / condition": "Singular analysis", "Output": "copy stem +N+SG", "Next state": "qf"},
])
display(fst_transitions)

def noun_fst(noun, number="PL"):
    '''Simple executable transducer behavior for the regular/exceptional rules in the question.'''
    noun = noun.strip()
    if not noun:
        raise ValueError("Noun cannot be empty.")
    if number.upper() == "SG":
        return noun + "+N+SG"
    if number.upper() != "PL":
        raise ValueError("number must be 'SG' or 'PL'")
    lower = noun.lower()
    suffix = "es" if lower.endswith(("s", "x", "z", "ch", "sh")) else "s"
    return noun + suffix + "+N+PL"

examples = ["cat", "book", "bus", "box", "quiz", "church", "brush"]
display(pd.DataFrame({
    "Noun": examples,
    "Singular lexical form": [noun_fst(w, "SG") for w in examples],
    "Plural lexical form": [noun_fst(w, "PL") for w in examples],
}))


,Current state,Input / condition,Output,Next state
0,q0,"End of noun; suffix does NOT end in s, x, z, c...",copy stem + s +N+PL,qf
1,q0,End of noun; suffix ends in s,copy stem + es +N+PL,qf
2,q0,End of noun; suffix ends in x,copy stem + es +N+PL,qf
3,q0,End of noun; suffix ends in z,copy stem + es +N+PL,qf
4,q0,End of noun; suffix ends in ch,copy stem + es +N+PL,qf
5,q0,End of noun; suffix ends in sh,copy stem + es +N+PL,qf
6,q0,Singular analysis,copy stem +N+SG,qf


,Noun,Singular lexical form,Plural lexical form
0,cat,cat+N+SG,cats+N+PL
1,book,book+N+SG,books+N+PL
2,bus,bus+N+SG,buses+N+PL
3,box,box+N+SG,boxes+N+PL
4,quiz,quiz+N+SG,quizes+N+PL
5,church,church+N+SG,churches+N+PL
6,brush,brush+N+SG,brushes+N+PL


**State interpretation:** `q0` is the start/stem-copying state; the transducer checks the final spelling of the input noun, selects the appropriate suffix, emits the grammatical tags, and reaches final state `qf`. In a fully expanded character-level FST, the stem-copying portion is represented by identity arcs, while suffix-specific paths emit `s` or `es`. This implementation encodes the suffix-selection rule given in the question.

## Q2. WordPiece algorithm — two merges

The corpus is entered exactly as five lines from the question. The implementation starts with character pieces and computes the WordPiece-style score:

`score(left, right) = pair_frequency / (frequency(left) × frequency(right))`

It merges the highest-scoring adjacent pair twice, then tokenizes **newest** using the learned pieces. WordPiece implementations can differ in initialization and tie-breaking; the notebook prints each merge so the assumption is transparent.

In [2]:
from collections import Counter
import re

corpus_lines = [
    "low lower low lower lowest",
    "low lowest low lowest",
    "low new",
    "new new newer newer",
    "new",
]
word_counts = Counter(word for line in corpus_lines for word in line.split())
display(pd.DataFrame(word_counts.items(), columns=["Word", "Frequency"]).sort_values("Word").reset_index(drop=True))

def train_wordpiece_merges(word_counts, n_merges=2):
    # Keep each word as a sequence of characters. Word frequency weights pair counts.
    tokenized = {word: list(word) for word in word_counts}
    merges = []

    for step in range(n_merges):
        pair_counts = Counter()
        piece_counts = Counter()
        for word, pieces in tokenized.items():
            freq = word_counts[word]
            for piece in pieces:
                piece_counts[piece] += freq
            for a, b in zip(pieces, pieces[1:]):
                pair_counts[(a, b)] += freq

        if not pair_counts:
            break

        scored = []
        for pair, count in pair_counts.items():
            score = count / (piece_counts[pair[0]] * piece_counts[pair[1]])
            scored.append((score, count, pair))
        # Deterministic tie-break: highest score, then highest count, then alphabetical pair.
        score, count, best_pair = sorted(
            scored, key=lambda item: (-item[0], -item[1], item[2][0], item[2][1])
        )[0]

        left, right = best_pair
        merged = left + right
        for word, pieces in tokenized.items():
            new_pieces, i = [], 0
            while i < len(pieces):
                if i + 1 < len(pieces) and pieces[i] == left and pieces[i+1] == right:
                    new_pieces.append(merged)
                    i += 2
                else:
                    new_pieces.append(pieces[i])
                    i += 1
            tokenized[word] = new_pieces
        merges.append({
            "Merge number": step + 1,
            "Pair merged": f"{left} + {right}",
            "New piece": merged,
            "Pair frequency": count,
            "WordPiece score": score,
        })
    return merges, tokenized

merges, learned_pieces = train_wordpiece_merges(word_counts, n_merges=2)
display(pd.DataFrame(merges))
print("Learned segmentation of corpus words:")
for word, pieces in learned_pieces.items():
    print(f"{word:8s} -> {' '.join(pieces)}")

def apply_greedy_pieces(word, pieces_vocab):
    # Greedy longest matching segmentation using learned pieces and original characters.
    vocab = set(pieces_vocab)
    result, i = [], 0
    while i < len(word):
        match = None
        for j in range(len(word), i, -1):
            candidate = word[i:j]
            if candidate in vocab:
                match = candidate
                break
        if match is None:
            # Character fallback makes the demonstration robust for unseen characters.
            match = word[i]
        result.append(match)
        i += len(match)
    return result

piece_vocab = set(ch for word in word_counts for ch in word)
for pieces in learned_pieces.values():
    piece_vocab.update(pieces)
print("Tokenization of 'newest':", apply_greedy_pieces("newest", piece_vocab))


,Word,Frequency
0,low,5
1,lower,2
2,lowest,3
3,new,4
4,newer,2


,Merge number,Pair merged,New piece,Pair frequency,WordPiece score
0,1,s + t,st,3,0.333333
1,2,l + o,lo,10,0.100000


Learned segmentation of corpus words:
low      -> lo w
lower    -> lo w e r
lowest   -> lo w e st
new      -> n e w
newer    -> n e w e r
Tokenization of 'newest': ['n', 'e', 'w', 'e', 'st']


**How to report Q2:** Write down the two rows in the merge table, including the score used to select each pair. The final line gives the segmentation of `newest`. If your class used ordinary BPE frequency-only merging instead of the normalized WordPiece score, the merges can differ; follow the algorithm specified by your instructor.

## Q3. TF-IDF for the four toy sentences

Use each sentence as one document. TF is the **raw count** of a word in that sentence. Unsmoothed IDF is `ln(N / df)`, where `N = 4` and `df` is the number of sentences containing the word. No start/end markers are added. Sentence closeness is measured by cosine similarity between TF-IDF vectors.

In [3]:
import numpy as np
from collections import Counter

sentences = {
    "S1": "cat dog animal",
    "S2": "cat cat pet",
    "S3": "dog pet animal",
    "S4": "cat dog pet",
}
tokens = {sid: sent.split() for sid, sent in sentences.items()}
vocabulary = sorted(set(w for words in tokens.values() for w in words))
N = len(sentences)

df = {word: sum(word in set(words) for words in tokens.values()) for word in vocabulary}
idf = {word: np.log(N / df[word]) for word in vocabulary}
tf = {sid: Counter(words) for sid, words in tokens.items()}
tfidf = pd.DataFrame(
    {sid: [tf[sid][word] * idf[word] for word in vocabulary] for sid in sentences},
    index=vocabulary
)

print("Document frequency:")
display(pd.DataFrame({"Word": vocabulary, "df": [df[w] for w in vocabulary],
                      "IDF = ln(4/df)": [idf[w] for w in vocabulary]}))
print("TF-IDF vectors (rows=words, columns=sentences):")
display(tfidf.round(6))

# Cosine similarity for every pair of sentence vectors.
vectors = {sid: tfidf[sid].to_numpy(dtype=float) for sid in sentences}
similarities = []
ids = list(sentences)
for i in range(len(ids)):
    for j in range(i+1, len(ids)):
        a, b = vectors[ids[i]], vectors[ids[j]]
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        cosine = float(np.dot(a, b) / denom) if denom else 0.0
        similarities.append({"Sentence pair": f"{ids[i]} and {ids[j]}", "Cosine similarity": cosine})
similarity_df = pd.DataFrame(similarities).sort_values("Cosine similarity", ascending=False).reset_index(drop=True)
display(similarity_df.round(6))
print("Closest pair:", similarity_df.iloc[0]["Sentence pair"])


Document frequency:


,Word,df,IDF = ln(4/df)
0,animal,2,0.693147
1,cat,3,0.287682
2,dog,3,0.287682
3,pet,3,0.287682


TF-IDF vectors (rows=words, columns=sentences):


,S1,S2,S3,S4
animal,0.693147,0.000000,0.693147,0.000000
cat,0.287682,0.575364,0.000000,0.287682
dog,0.287682,0.000000,0.287682,0.287682
pet,0.000000,0.287682,0.287682,0.287682


,Sentence pair,Cosine similarity
0,S1 and S3,0.871882
1,S2 and S4,0.774597
2,S3 and S4,0.413308
3,S1 and S4,0.413308
4,S1 and S2,0.320147
5,S2 and S3,0.160074


Closest pair: S1 and S3


## Q4. Bernoulli Naive Bayes spam classification

The three features are binary indicators for `free`, `win`, and `offer`. For Bernoulli Naive Bayes, both presence and absence probabilities are used. First, compute the class priors and conditional probabilities from the seven emails. Then score E8 = `(0, 0, 0)` without smoothing, as the question does not specify smoothing.

In [4]:
emails = pd.DataFrame([
    ["E1", 1, 1, 1, "Spam"],
    ["E2", 1, 1, 0, "Spam"],
    ["E3", 1, 0, 1, "Spam"],
    ["E4", 0, 1, 1, "Spam"],
    ["E5", 0, 0, 1, "NotSpam"],
    ["E6", 0, 1, 0, "NotSpam"],
    ["E7", 1, 0, 0, "NotSpam"],
], columns=["Email", "free", "win", "offer", "Class"])
display(emails)

features = ["free", "win", "offer"]
class_counts = emails["Class"].value_counts()
priors = {c: class_counts[c] / len(emails) for c in class_counts.index}
print("Class priors:", priors)

# P(feature=1 | class) and P(feature=0 | class)
conditional = {}
for c in ["Spam", "NotSpam"]:
    subset = emails[emails["Class"] == c]
    conditional[c] = {}
    for feature in features:
        p1 = float(subset[feature].mean())
        conditional[c][feature] = {1: p1, 0: 1-p1}

conditional_rows = []
for c in ["Spam", "NotSpam"]:
    for feature in features:
        conditional_rows.append({
            "Class": c, "Feature": feature,
            "P(feature=1 | class)": conditional[c][feature][1],
            "P(feature=0 | class)": conditional[c][feature][0],
        })
display(pd.DataFrame(conditional_rows))

email8 = {"free": 0, "win": 0, "offer": 0}
scores = {}
for c in ["Spam", "NotSpam"]:
    likelihood = 1.0
    for feature in features:
        likelihood *= conditional[c][feature][email8[feature]]
    scores[c] = priors[c] * likelihood

print("E8 features:", email8)
print("Unnormalized posterior scores:", scores)
prediction = max(scores, key=scores.get)
print("Prediction without smoothing:", prediction)


,Email,free,win,offer,Class
0,E1,1,1,1,Spam
1,E2,1,1,0,Spam
2,E3,1,0,1,Spam
3,E4,0,1,1,Spam
4,E5,0,0,1,NotSpam
5,E6,0,1,0,NotSpam
6,E7,1,0,0,NotSpam


Class priors: {'Spam': np.float64(0.5714285714285714), 'NotSpam': np.float64(0.42857142857142855)}


,Class,Feature,P(feature=1 | class),P(feature=0 | class)
0,Spam,free,0.750000,0.250000
1,Spam,win,0.750000,0.250000
2,Spam,offer,0.750000,0.250000
3,NotSpam,free,0.333333,0.666667
4,NotSpam,win,0.333333,0.666667
5,NotSpam,offer,0.333333,0.666667


E8 features: {'free': 0, 'win': 0, 'offer': 0}
Unnormalized posterior scores: {'Spam': np.float64(0.008928571428571428), 'NotSpam': np.float64(0.126984126984127)}
Prediction without smoothing: NotSpam


**Interpretation:** For E8 = `(0, 0, 0)`, the likelihood for each class is the product of the three absence probabilities. The Spam score is `(4/7) × (1/4) × (1/4) × (1/4) = 1/112 ≈ 0.00893`. The NotSpam score is `(3/7) × (2/3) × (2/3) × (2/3) = 8/63 ≈ 0.12698`. Since the NotSpam score is larger, the prediction is **NotSpam**. The data include one Spam example with `offer = 0`, so the unsmoothed probability is not zero.

## Q5. Kneser–Ney smoothing

The five sentences are used without start or end markers. For standard interpolated bigram Kneser–Ney:

`P_KN(w | h) = max(c(h,w)-d, 0) / c(h,*) + λ(h) P_cont(w)`

where `λ(h) = d × N1+(h,*) / c(h,*)` and `P_cont(w) = N1+(*,w) / N1+(*,*)`. Use `d = 0.5`. `N1+(*,w)` counts distinct predecessor words for `w`; `N1+(*,*)` is the number of distinct bigram types.

In [5]:
kn_sentences = [
    "I like NLP",
    "I like AI",
    "You like NLP",
    "You love AI",
    "We love NLP",
]
bigram_counts = Counter()
for sentence in kn_sentences:
    words = sentence.split()
    for a, b in zip(words, words[1:]):
        bigram_counts[(a, b)] += 1

predecessors = {}
successors = {}
for (a, b), count in bigram_counts.items():
    predecessors.setdefault(b, set()).add(a)
    successors.setdefault(a, set()).add(b)

distinct_bigram_types = len(bigram_counts)
continuation_prob = {
    word: len(predecessors.get(word, set())) / distinct_bigram_types
    for word in set(b for _, b in bigram_counts)
}

print("Bigram counts:")
display(pd.DataFrame([
    {"Bigram": f"{a} → {b}", "Count": c}
    for (a, b), c in sorted(bigram_counts.items())
]))
print("Number of distinct bigram types N1+(*,*):", distinct_bigram_types)
print("Distinct predecessor counts N1+(*,w):",
      {w: len(predecessors.get(w, set())) for w in sorted(continuation_prob)})
print("Continuation probabilities:", continuation_prob)

d = 0.5

def kneser_ney_probability(word, history):
    c_hw = bigram_counts[(history, word)]
    c_h = sum(count for (h, _), count in bigram_counts.items() if h == history)
    distinct_followers = len(successors.get(history, set()))
    if c_h == 0:
        return continuation_prob.get(word, 0.0)
    direct = max(c_hw - d, 0) / c_h
    lamb = d * distinct_followers / c_h
    p_cont = continuation_prob.get(word, 0.0)
    return direct + lamb * p_cont

queries = [("AI", "love"), ("NLP", "like")]
kn_rows = []
for word, history in queries:
    c_hw = bigram_counts[(history, word)]
    c_h = sum(count for (h, _), count in bigram_counts.items() if h == history)
    t_h = len(successors.get(history, set()))
    direct = max(c_hw - d, 0) / c_h
    lamb = d * t_h / c_h
    pcont = continuation_prob.get(word, 0.0)
    kn_rows.append({
        "Requested probability": f"P_KN({word} | {history})",
        "c(history, word)": c_hw,
        "c(history, *)": c_h,
        "N1+(history, *)": t_h,
        "Direct term": direct,
        "Lambda(history)": lamb,
        "P_cont(word)": pcont,
        "Interpolated probability": direct + lamb*pcont,
    })
display(pd.DataFrame(kn_rows).round(6))


Bigram counts:


,Bigram,Count
0,I → like,2
1,We → love,1
2,You → like,1
3,You → love,1
4,like → AI,1
5,like → NLP,2
6,love → AI,1
7,love → NLP,1


Number of distinct bigram types N1+(*,*): 8
Distinct predecessor counts N1+(*,w): {'AI': 2, 'NLP': 2, 'like': 2, 'love': 2}
Continuation probabilities: {'AI': 0.25, 'like': 0.25, 'love': 0.25, 'NLP': 0.25}


,Requested probability,"c(history, word)","c(history, *)","N1+(history, *)",Direct term,Lambda(history),P_cont(word),Interpolated probability
0,P_KN(AI | love),1,2,2,0.25,0.500000,0.25,0.375000
1,P_KN(NLP | like),2,3,2,0.50,0.333333,0.25,0.583333
